In [1]:
import os, glob, re, random, time, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from PIL import Image

warnings.filterwarnings('ignore')

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT    = '/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset'
BASE_H  = 176
BASE_W  = 512
N_CLASSES = 6
SEED    = 42
EPOCHS  = 45
BATCH   = 64
LR      = 1e-3
GAP     = 10
DUP_SZ  = 32

CLASS_MAP = {
    '3_long_blade_rotor':     '3_long_blade_rotor',
    '3_short_blade_rotor_1':  '3_short_blade_rotor',
    '3_short_blade_rotor_2':  '3_short_blade_rotor',
    'Bird':                   'Bird',
    'Bird+mini-helicopter_1': 'Bird+mini-helicopter',
    'Bird+mini-helicopter_2': 'Bird+mini-helicopter',
    'RC plane_1':             'RC_plane',
    'RC plane_2':             'RC_plane',
    'drone_1':                'drone',
    'drone_2':                'drone',
}
CLASSES = sorted(set(CLASS_MAP.values()))
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}


def numeric_key(p):
    nums = re.findall(r'\d+', os.path.basename(p))
    return int(nums[0]) if nums else 0


def list_images(folder):
    exts = ('*.png','*.jpg','*.jpeg','*.PNG','*.JPG','*.JPEG')
    out = []
    for e in exts:
        out += glob.glob(os.path.join(folder, '**', e), recursive=True)
    return out


def autocrop_resize(path):
    img = Image.open(path).convert('L')
    arr = np.asarray(img, dtype=np.float32)
    mask = arr < 240
    if mask.any():
        rows = np.where(mask.any(axis=1))[0]
        cols = np.where(mask.any(axis=0))[0]
        arr = arr[rows[0]:rows[-1]+1, cols[0]:cols[-1]+1]
    out = np.asarray(
        Image.fromarray(arr.astype(np.uint8)).resize((BASE_W, BASE_H), Image.BILINEAR),
        dtype=np.uint8)
    return out


print('Enumerating files in filename order...')
paths, labels, leaf_of = [], [], []
for top in sorted(os.listdir(ROOT)):
    if top in CLASS_MAP:
        c = CLS2IDX[CLASS_MAP[top]]
        files = sorted(list_images(os.path.join(ROOT, top)), key=lambda p: (numeric_key(p), p))
        for f in files:
            paths.append(f); labels.append(c); leaf_of.append(top)
labels = np.array(labels, dtype=np.int64)
N = len(paths)
print(f'Total images: {N}')

print('Loading images...')
bdata = np.zeros((N, BASE_H, BASE_W), dtype=np.uint8)
for i, p in enumerate(paths):
    bdata[i] = autocrop_resize(p)
print('Loaded:', bdata.shape)

idx_all = np.arange(N)
r_tr, r_tmp = train_test_split(idx_all, test_size=0.2, stratify=labels, random_state=SEED)
r_va, r_te = train_test_split(r_tmp, test_size=0.5, stratify=labels[r_tmp], random_state=SEED)

b_tr, b_va, b_te = [], [], []
for c in range(N_CLASSES):
    ci = idx_all[labels == c]
    n = len(ci)
    n_tr = int(round(n * 0.8))
    n_va = int(round(n * 0.1))
    tr = ci[:max(n_tr - GAP, 1)]
    va = ci[n_tr + GAP : n_tr + n_va]
    te = ci[n_tr + n_va + GAP :]
    b_tr += list(tr); b_va += list(va); b_te += list(te)
b_tr, b_va, b_te = np.array(b_tr), np.array(b_va), np.array(b_te)

print(f'Random split  tr/va/te: {len(r_tr)}/{len(r_va)}/{len(r_te)}')
print(f'Blocked split tr/va/te: {len(b_tr)}/{len(b_va)}/{len(b_te)}  (GAP={GAP} dropped per seam)')


def dup_features(arr):
    t = torch.from_numpy(arr).float().unsqueeze(1)
    t = F.interpolate(t, size=(DUP_SZ, DUP_SZ), mode='bilinear', align_corners=False)
    v = t.view(t.size(0), -1)
    v = v - v.mean(dim=1, keepdim=True)
    v = v / (v.norm(dim=1, keepdim=True) + 1e-8)
    return v.numpy().astype(np.float32)


def knn_eval(F_all, tr_idx, te_idx, lab):
    Xtr, Xte = F_all[tr_idx], F_all[te_idx]
    sims = Xte @ Xtr.T
    nn_idx = sims.argmax(axis=1)
    nn_sim = sims.max(axis=1)
    pred = lab[tr_idx][nn_idx]
    acc = accuracy_score(lab[te_idx], pred)
    return acc, nn_sim


def same_class_pair_sim(F_all, lab, n_pairs=5000):
    sims = []
    rng = np.random.RandomState(0)
    for _ in range(n_pairs):
        c = rng.randint(N_CLASSES)
        ci = np.where(lab == c)[0]
        a, b = ci[rng.randint(len(ci))], ci[rng.randint(len(ci))]
        if a != b:
            sims.append(float(F_all[a] @ F_all[b]))
    return np.array(sims)


def consecutive_sim(F_all, lab, leaf):
    sims = []
    for i in range(len(lab) - 1):
        if lab[i] == lab[i+1] and leaf[i] == leaf[i+1]:
            sims.append(float(F_all[i] @ F_all[i+1]))
    return np.array(sims)


print('\nComputing duplicate features...')
Fd = dup_features(bdata)

print('\n' + '=' * 70)
print('DIAGNOSTIC A: RAW-PIXEL 1-NN (no learning)')
print('=' * 70)
knn_rand_acc, knn_rand_sim = knn_eval(Fd, r_tr, r_te, labels)
knn_blk_acc, knn_blk_sim = knn_eval(Fd, b_tr, b_te, labels)
print(f'1-NN accuracy  random split : {knn_rand_acc*100:.2f}%  (median NN sim {np.median(knn_rand_sim):.3f})')
print(f'1-NN accuracy  blocked split: {knn_blk_acc*100:.2f}%  (median NN sim {np.median(knn_blk_sim):.3f})')

print('\n' + '=' * 70)
print('DIAGNOSTIC B: NEAR-DUPLICATE BOUNDARY SCAN (random split)')
print('=' * 70)
for thr in (0.90, 0.95, 0.97, 0.99):
    frac = (knn_rand_sim >= thr).mean()
    print(f'  test images with a train neighbor >= {thr:.2f} sim : {frac*100:5.1f}%')
base_pair = same_class_pair_sim(Fd, labels)
cons = consecutive_sim(Fd, labels, leaf_of)
print(f'  mean test->train nearest sim     : {knn_rand_sim.mean():.3f}')
print(f'  mean random same-class pair sim  : {base_pair.mean():.3f}')
print(f'  mean consecutive same-leaf sim   : {cons.mean():.3f}  (n={len(cons)})')


class ConvBNSiLU(nn.Module):
    def __init__(self, ci, co, k=3, s=1):
        super().__init__()
        self.b = nn.Sequential(nn.Conv2d(ci, co, k, s, k//2, bias=False),
                               nn.BatchNorm2d(co), nn.SiLU(inplace=True))
    def forward(self, x): return self.b(x)


class Conv1dBNSiLU(nn.Module):
    def __init__(self, ci, co, k=5, s=2):
        super().__init__()
        self.b = nn.Sequential(nn.Conv1d(ci, co, k, s, k//2, bias=False),
                               nn.BatchNorm1d(co), nn.SiLU(inplace=True))
    def forward(self, x): return self.b(x)


def make_head(in_dim, n, dropout=0.3):
    h = max(in_dim // 2, 32)
    return nn.Sequential(nn.Linear(in_dim, h), nn.BatchNorm1d(h),
                         nn.SiLU(inplace=True), nn.Dropout(dropout), nn.Linear(h, n))


class VelocityOnly(nn.Module):
    def __init__(self, n=N_CLASSES, w=32, oc=64):
        super().__init__()
        self.stem = nn.Sequential(ConvBNSiLU(1, 16, 3, 2), ConvBNSiLU(16, w, 3, 2))
        self.vel = nn.Sequential(Conv1dBNSiLU(w, oc, 5, 2), Conv1dBNSiLU(oc, oc, 5, 2))
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.head = make_head(oc, n)
    def forward(self, x):
        f = self.stem(x)
        v = self.gap(self.vel(f.mean(dim=3))).flatten(1)
        return self.head(v)


class DS(Dataset):
    def __init__(self, idx, train):
        self.idx = idx; self.train = train
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        img = bdata[j].astype(np.float32) / 255.0
        if self.train:
            if random.random() < 0.5:
                img = img[:, ::-1].copy()
            if random.random() < 0.5:
                img = np.clip(img * random.uniform(0.9, 1.1), 0.0, 1.0)
            if random.random() < 0.3:
                h0 = random.randint(0, BASE_H - 20)
                img[h0:h0+random.randint(5, 20), :] = 0.0
            if random.random() < 0.3:
                w0 = random.randint(0, BASE_W - 25)
                img[:, w0:w0+random.randint(5, 25)] = 0.0
        return torch.from_numpy(img).unsqueeze(0), int(labels[j])


def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def train_eval(tr_idx, va_idx, te_idx, tag):
    set_seed(SEED)
    m = VelocityOnly().to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-2)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    scl = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
    tl = DataLoader(DS(tr_idx, True), batch_size=BATCH, shuffle=True,
                    num_workers=2, pin_memory=True, drop_last=True)
    vl = DataLoader(DS(va_idx, False), batch_size=BATCH, shuffle=False,
                    num_workers=2, pin_memory=True)
    best, best_state = 0.0, None
    for ep in range(EPOCHS):
        m.train()
        for x, y in tl:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad()
            with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
                loss = crit(m(x), y)
            scl.scale(loss).backward(); scl.step(opt); scl.update()
        sch.step()
        m.eval(); c = t = 0
        with torch.no_grad():
            for x, y in vl:
                x, y = x.to(DEVICE), y.to(DEVICE)
                c += (m(x).argmax(1) == y).sum().item(); t += y.size(0)
        if c / t > best:
            best = c / t
            best_state = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
    m.load_state_dict(best_state); m.eval()
    tel = DataLoader(DS(te_idx, False), batch_size=BATCH, shuffle=False,
                     num_workers=2, pin_memory=True)
    yt, yp = [], []
    with torch.no_grad():
        for x, y in tel:
            yp.append(m(x.to(DEVICE)).argmax(1).cpu().numpy()); yt.append(y.numpy())
    acc = accuracy_score(np.concatenate(yt), np.concatenate(yp))
    print(f'  velocity-only [{tag}] test accuracy: {acc*100:.2f}%')
    return acc


print('\n' + '=' * 70)
print('DIAGNOSTIC C: VELOCITY-ONLY, RANDOM vs BLOCKED SPLIT')
print('=' * 70)
acc_rand = train_eval(r_tr, r_va, r_te, 'random')
acc_blk = train_eval(b_tr, b_va, b_te, 'blocked')
drop = (acc_rand - acc_blk) * 100

print('\n' + '=' * 70)
print('VERDICT')
print('=' * 70)
print(f'  Raw-pixel 1-NN (random)        : {knn_rand_acc*100:.2f}%')
print(f'  Raw-pixel 1-NN (blocked)       : {knn_blk_acc*100:.2f}%')
print(f'  Velocity-only (random)         : {acc_rand*100:.2f}%')
print(f'  Velocity-only (blocked)        : {acc_blk*100:.2f}%')
print(f'  Accuracy drop random->blocked  : {drop:.2f} points')
print(f'  Near-dup fraction (>=0.97 sim) : {(knn_rand_sim>=0.97).mean()*100:.1f}%')
leak_signals = 0
if knn_rand_acc > 0.90: leak_signals += 1
if (knn_rand_sim >= 0.97).mean() > 0.20: leak_signals += 1
if drop > 5.0: leak_signals += 1
if knn_rand_acc - knn_blk_acc > 0.10: leak_signals += 1
print(f'\n  Leakage signals triggered: {leak_signals}/4')
if leak_signals >= 2:
    print('  => LEAKAGE LIKELY. The random split shares near-duplicate frames')
    print('     across train/test. Use the blocked (or recording-grouped) split')
    print('     as the honest evaluation protocol.')
else:
    print('  => NO STRONG LEAKAGE SIGNAL. High accuracy appears to reflect a')
    print('     genuinely separable task rather than train/test duplication.')

Enumerating files in filename order...
Total images: 4849
Loading images...
Loaded: (4849, 176, 512)
Random split  tr/va/te: 3879/485/485
Blocked split tr/va/te: 3819/426/424  (GAP=10 dropped per seam)

Computing duplicate features...

DIAGNOSTIC A: RAW-PIXEL 1-NN (no learning)
1-NN accuracy  random split : 64.33%  (median NN sim 0.921)
1-NN accuracy  blocked split: 39.62%  (median NN sim 0.906)

DIAGNOSTIC B: NEAR-DUPLICATE BOUNDARY SCAN (random split)
  test images with a train neighbor >= 0.90 sim :  63.7%
  test images with a train neighbor >= 0.95 sim :  12.8%
  test images with a train neighbor >= 0.97 sim :   0.6%
  test images with a train neighbor >= 0.99 sim :   0.0%
  mean test->train nearest sim     : 0.896
  mean random same-class pair sim  : 0.630
  mean consecutive same-leaf sim   : 0.801  (n=4839)

DIAGNOSTIC C: VELOCITY-ONLY, RANDOM vs BLOCKED SPLIT
  velocity-only [random] test accuracy: 98.97%
  velocity-only [blocked] test accuracy: 89.15%

VERDICT
  Raw-pixel 1-NN 